## Import libraries

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
import config
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline

## Functions

In [2]:
def create_preprocessing():
    # Fit all scaling statistics on the training fold only
    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
    ])

    return ColumnTransformer([
        ("numeric", numerical_pipeline,
         make_column_selector(pattern=r"^(?!AssetClass_|Ticker_|Regime_).+")),
        ("indicators", "passthrough",
         make_column_selector(pattern=r"^(?:AssetClass_|Ticker_|Regime_).+")),
    ], sparse_threshold=0.0)


def create_classifier(params=None):
    if params is None:
        params = best_lr_params  # Selected by grid search or loaded from JSON

    model_params = {
        "solver": "liblinear",
        "l1_ratio": 0.0,  # pure L2 regularization
        "max_iter": 500,
        "random_state": int(config.RANDOM_STATE),
        **params,
    }
    return LogisticRegression(**model_params)


def create_model(params=None):
    return Pipeline([
        ("preprocessing", create_preprocessing()),
        ("classifier", create_classifier(params=params)),
    ])

def predict_probabilities(model, features):
    if isinstance(model, Pipeline):
        processed = model.named_steps["preprocessing"].transform(features)
        classifier = model.named_steps["classifier"]
    else:
        processed = features
        classifier = model

    if not np.array_equal(classifier.classes_, [0, 1]):
        raise ValueError("Expected fitted Logistic Regression with classes [0, 1].")

    scores = np.einsum(
        "ij,j->i",
        np.asarray(processed),
        classifier.coef_[0],
        optimize=False,
    ) + classifier.intercept_[0]
    return expit(scores)


def X(data, columns):
    return data.loc[:, columns].replace([np.inf, -np.inf], np.nan)

def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan

def safe_ap(y, p):
    return float(average_precision_score(y, p)) if np.unique(y).size == 2 else np.nan

def direction_metrics(data, threshold):
    y = data[TARGET].to_numpy(dtype=int)
    p = data["Probability"].to_numpy(dtype=float)
    predicted_positive = p >= threshold
    selected = int(predicted_positive.sum())
    true_positive = int(np.sum(predicted_positive & (y == 1)))
    false_positive = int(np.sum(predicted_positive & (y == 0)))
    true_negative = int(np.sum(~predicted_positive & (y == 0)))
    false_negative = int(np.sum(~predicted_positive & (y == 1)))
    precision = true_positive / selected if selected else np.nan
    base = float(y.mean())
    return {
        "Observations": len(y),
        "Always_Positive_Precision": base,
        "Precision": precision,
        "Precision_Lift_pp": 100.0 * (precision - base) if selected else np.nan,
        "Recall": true_positive / (true_positive + false_negative) if np.any(y == 1) else np.nan,
        "Specificity": true_negative / (true_negative + false_positive) if np.any(y == 0) else np.nan,
        "Balanced_Accuracy": float(balanced_accuracy_score(y, predicted_positive)),
        "Positive_Prediction_Rate": selected / len(y),
        "Predicted_Positive_Count": selected,
        "ROC_AUC": safe_auc(y, p),
        "Average_Precision": safe_ap(y, p),
    }

def screening_summary(part):
    n_selected = int(part["Selected_Assets"].sum())
    n_correct = int(part["Correct_Positives"].sum())
    total_dates = len(part)
    n_active = int((part["Selected_Assets"] > 0).sum())
    precision = n_correct / n_selected if n_selected else np.nan
    matched_baseline = (
        float((part["Selected_Assets"] * part["Same_Date_Baseline"]).sum() / n_selected)
        if n_selected else np.nan
    )
    return {
        "Dates": total_dates,
        "Selected_Asset_Dates": n_selected,
        "Precision": precision,
        "Matched_Date_Baseline": matched_baseline,
        "Precision_Lift_pp": 100 * (precision - matched_baseline) if n_selected else np.nan,
        "Mean_Assets": n_selected / total_dates,
        "Min_Assets": int(part["Selected_Assets"].min()),
        "Max_Assets": int(part["Selected_Assets"].max()),
        "Cash_Only_Dates": total_dates - n_active,
    }

## Load features and define feature sets

In [3]:
# Output folder and model-selection mode
OUTPUT_DIR = Path("classification_results/logistic_regression")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not isinstance(config.HYPERPARAMETER_TUNING, bool):
    raise TypeError("HYPERPARAMETER_TUNING must be True or False.")

ALGORITHMS = ["Logistic Regression"]
selected_config_path = OUTPUT_DIR / "selected_config.json"
loaded_config = None
best_lr_params = None

if not config.HYPERPARAMETER_TUNING:
    if not selected_config_path.is_file():
        raise FileNotFoundError(
            f"Hyperparameter tuning is disabled, but {selected_config_path} does not exist. "
            "Run once with HYPERPARAMETER_TUNING=True to save a validated configuration."
        )
    with selected_config_path.open("r", encoding="utf-8") as file:
        loaded_config = json.load(file)
    if loaded_config.get("algorithm") != "Logistic Regression":
        raise ValueError("Saved selected_config.json is not a Logistic Regression configuration.")
    if not isinstance(loaded_config.get("model_params"), dict):
        raise ValueError("Saved configuration is missing model_params.")
    if not isinstance(loaded_config.get("screening"), dict):
        raise ValueError("Saved configuration is missing screening.")
    best_lr_params = dict(loaded_config["model_params"])
    print(f"Hyperparameter tuning skipped. Loaded saved model and screening configuration from {selected_config_path}")
else:
    print("Hyperparameter tuning enabled: Logistic Regression C-grid search will run on validation data.")

# Set features folder
FEATURES_DATA_DIR = config.INPUT_DATA_DIR.parent / "features_data"

# Load asset classes from the investable universe
universe = pd.read_csv(config.UNIVERSE_FILE)

# Read features files
df = pd.read_csv(FEATURES_DATA_DIR / "combined_features.csv", parse_dates=["Date"])
regimes = pd.read_csv(FEATURES_DATA_DIR / "market_regimes.csv", parse_dates=["Date"])

# Merge with confirmed regime label
df = df.merge(
    regimes[["Date", "Market_Regime"]],
    on="Date",
    how="left",
    validate="many_to_one"
)

df = df.sort_values(["Date", "Ticker"]).reset_index(drop=True)

# Identify dates with missing regime labels
missing_dates = (df.loc[df["Market_Regime"].isna(), "Date"].drop_duplicates().sort_values())
missing_count = df["Market_Regime"].isna().sum()

# Forward fill using the last available confirmed regime
df["Market_Regime"] = df["Market_Regime"].ffill()

# Print dates and assigned regime labels
filled_dates = (
    df.loc[df["Date"].isin(missing_dates), ["Date", "Market_Regime"]]
    .drop_duplicates()
    .sort_values("Date")
)

print(f"Forward-filled regime dates: {len(missing_dates)}")

# Final checks
print("\nDataset shape:", df.shape)
print("Missing regime labels:", df["Market_Regime"].isna().sum())
print("Duplicate asset-dates:", df.duplicated(["Date", "Ticker"]).sum())
print(df["Target_60d"].value_counts(normalize=True))

asset_classes = universe[["Ticker", "Asset Class"]].drop_duplicates(subset="Ticker")

# Merge asset classes into the modeling dataset
df = df.merge(
    asset_classes,
    on="Ticker",
    how="left",
    validate="many_to_one"
)

# Encode asset classes
asset_class_dummies = pd.get_dummies(df["Asset Class"], prefix="AssetClass", dtype=int)
df = pd.concat([df, asset_class_dummies], axis=1)
asset_class_features = asset_class_dummies.columns.tolist()

print("\nAsset-class features:")
print(asset_class_features)

# One-hot encode ticker identity
ticker_dummies = pd.get_dummies(df["Ticker"], prefix="Ticker", dtype=int)
df = pd.concat([df, ticker_dummies], axis=1)
ticker_features = ticker_dummies.columns.tolist()

# Define target label
TARGET = "Target_60d"

# Columns exclude from features
EXCLUDE = ["Date", "Ticker", "Forward_Return_60d", "Target_60d", "Market_Regime", "Asset Class"]

# Define feature set
base_features = [c for c in df.columns if c not in EXCLUDE and not c.startswith("Regime_") and not c.startswith("Ticker_")]

# Use one-hot encoding for confirmed regimes
regime_dummies = pd.get_dummies(df["Market_Regime"], prefix="Regime", dtype=int)
df = pd.concat([df, regime_dummies], axis=1)
regime_features = regime_dummies.columns.tolist()

# Regime specifications
features_without_regime = base_features
features_with_regime = base_features + regime_features

# Ticker-level specifications
features_with_tickers = base_features + ticker_features
features_with_tickers_regime = (base_features + ticker_features + regime_features)

assert not df.duplicated(["Date", "Ticker"]).any()
assert df["Asset Class"].notna().all()
assert df[TARGET].notna().any()
assert not df["Market_Regime"].isna().any()
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in features_with_tickers_regime)

SPECIFICATIONS = {
    "Without Regimes": features_with_tickers,
    "With Regimes": features_with_tickers_regime,
}

for name, features in SPECIFICATIONS.items():
    assert len(features) == len(set(features)), name
    assert not set(["Date", "Ticker", TARGET, "Forward_Return_60d"]).intersection(features)

Hyperparameter tuning skipped. Loaded saved model and screening configuration from classification_results\logistic_regression\selected_config.json
Forward-filled regime dates: 36

Dataset shape: (215430, 256)
Missing regime labels: 0
Duplicate asset-dates: 0
Target_60d
1    0.615105
0    0.384895
Name: proportion, dtype: float64

Asset-class features:
['AssetClass_Commodity', 'AssetClass_Crypto', 'AssetClass_Equity', 'AssetClass_Fixed Income', 'AssetClass_Foreign Exchange', 'AssetClass_Volatility']


## Walk-forward validation

In [4]:
# Validation period and rebalance frequency
VALIDATION_START = pd.Timestamp(config.VALIDATION_START)
TEST_START = pd.Timestamp(config.TEST_START)
REBALANCE_STEP = config.REBALANCE_FREQUENCY

# Calculate when each asset's 60-session target becomes known
df = df.sort_values(["Ticker", "Date"], kind="stable").reset_index(drop=True).copy()
df["_Target_End_Date"] = (
    df.groupby("Ticker", sort=False)["Date"].shift(-config.FORWARD_HORIZON)
)

valid_target = (
    df[TARGET].isin([0, 1])
    & df["Forward_Return_60d"].notna()
    & df["_Target_End_Date"].notna()
)

# Use the dates already available in the merged dataframe
trading_dates = pd.DatetimeIndex(sorted(df["Date"].dropna().unique()))

# Anchor the schedule to the first 2022 rebalance date
test_anchor = trading_dates[trading_dates >= TEST_START][0]
anchor_pos = trading_dates.get_loc(test_anchor)

# Generate 60-session rebalance dates backwards from 2022
positions = np.arange(anchor_pos - REBALANCE_STEP, -1, -REBALANCE_STEP)
validation_dates = trading_dates[positions]
validation_dates = validation_dates[
    (validation_dates >= VALIDATION_START)
    & (validation_dates < TEST_START)
].sort_values()

# Keep only row indices
walk_forward_splits = []
validation_summary = []
skipped_dates = []

for rebalance_date in validation_dates:
    train_idx = df.index[
        (df["Date"] < rebalance_date)
        & (df["_Target_End_Date"] < rebalance_date)
        & valid_target
    ].to_numpy()

    val_idx = df.index[df["Date"] == rebalance_date].to_numpy()

    # Require complete, known outcomes for the rebalance date
    known = valid_target.loc[val_idx] & (df.loc[val_idx, "_Target_End_Date"] < TEST_START)
    if len(val_idx) == 0 or not known.all():
        skipped_dates.append(rebalance_date)
        continue

    assert len(train_idx) > 0, rebalance_date
    assert df.loc[train_idx, TARGET].nunique() == 2, rebalance_date
    assert df.loc[train_idx, "_Target_End_Date"].max() < rebalance_date
    assert df.loc[val_idx, "Date"].nunique() == 1
    assert df.loc[val_idx, "_Target_End_Date"].max() < TEST_START

    walk_forward_splits.append((rebalance_date, train_idx, val_idx))
    validation_summary.append({
        "Rebalance_Date": rebalance_date,
        "Train_Start": df.loc[train_idx, "Date"].min(),
        "Train_End": df.loc[train_idx, "Date"].max(),
        "Train_Obs": len(train_idx),
        "Val_Obs": len(val_idx),
        "Val_Baseline": df.loc[val_idx, TARGET].mean(),
    })

validation_assets = pd.concat(
    [df.loc[val_idx, ["Date", "Ticker"]] for _, _, val_idx in walk_forward_splits],
    ignore_index=True,
)
assert not validation_assets.duplicated(["Date", "Ticker"]).any()

print("Scheduled validation dates:", len(validation_dates))
print("Completed validation dates:", len(walk_forward_splits))
print("Excluded immature/incomplete dates:", len(skipped_dates))
display(pd.DataFrame(validation_summary))

Scheduled validation dates: 25
Completed validation dates: 24
Excluded immature/incomplete dates: 1


,Rebalance_Date,Train_Start,Train_End,Train_Obs,Val_Obs,Val_Baseline
0,2016-01-20,2008-01-02,2015-10-21,86587,48,0.958333
1,2016-04-15,2008-01-02,2016-01-19,89467,48,0.833333
2,2016-07-12,2008-01-02,2016-04-14,92347,48,0.541667
3,2016-10-05,2008-01-02,2016-07-11,95227,48,0.333333
4,2016-12-30,2008-01-02,2016-10-04,98107,48,0.895833
5,2017-03-29,2008-01-02,2016-12-29,100987,48,0.833333
6,2017-06-23,2008-01-02,2017-03-28,103867,48,0.875000
7,2017-09-19,2008-01-02,2017-06-22,106747,48,0.666667
8,2017-12-13,2008-01-02,2017-09-18,109627,48,0.625000
9,2018-03-13,2008-01-02,2017-12-12,112507,48,0.541667


## Hyperparameter search space

In [5]:
# Run the full C grid when tuning is enabled
if config.HYPERPARAMETER_TUNING:
    TUNING_SPECIFICATION = "With Regimes"
    LR_PARAM_SPACE = {
        "C": [1, 0.1, 0.01, 0.001, 0.0001],
    }
    parameter_sets = list(ParameterGrid(LR_PARAM_SPACE))
    N_ITER = len(parameter_sets)

    print(f"Logistic Regression parameter configurations: {N_ITER}")
    print(f"Walk-forward validation dates: {len(walk_forward_splits)}")
    print(f"Search model fits: {N_ITER * len(walk_forward_splits)}")

## Hyperparameter grid search

In [6]:
if config.HYPERPARAMETER_TUNING:
    tuning_features = SPECIFICATIONS[TUNING_SPECIFICATION]
    search_daily_rows = []

    # Preprocess each walk-forward training window only once,
    # then evaluate all C values on the same transformed data
    with tqdm(
        total=len(parameter_sets) * len(walk_forward_splits),
        desc="Logistic Regression C grid search",
        unit="fit",
    ) as progress:

        for rebalance_date, train_idx, val_idx in walk_forward_splits:

            y_train = df.loc[train_idx, TARGET].to_numpy(dtype=int)
            y_val = df.loc[val_idx, TARGET].to_numpy(dtype=int)

            X_train = X(df.loc[train_idx, tuning_features], tuning_features)
            X_val = X(df.loc[val_idx, tuning_features], tuning_features)

            # Fit preprocessing on training observations only
            preprocessing = create_preprocessing()

            train_processed = preprocessing.fit_transform(X_train)
            val_processed = preprocessing.transform(X_val)

            del X_train, X_val

            # Evaluate each C value
            for trial, params in enumerate(parameter_sets, start=1):

                model = create_classifier(params=params)
                model.fit(train_processed, y_train)

                p_train = predict_probabilities(model, train_processed)
                p_val = predict_probabilities(model, val_processed)

                train_auc = safe_auc(y_train, p_train)
                val_auc = safe_auc(y_val, p_val)

                search_daily_rows.append({
                    "Trial": trial,
                    "Rebalance_Date": rebalance_date,
                    "Train_Observations": len(y_train),
                    "Validation_Observations": len(y_val),
                    "Train_ROC_AUC": train_auc,
                    "Val_ROC_AUC": val_auc,
                    "Val_Average_Precision": safe_ap(y_val, p_val),
                    "Val_Log_Loss": float(
                        log_loss(y_val, p_val, labels=[0, 1])
                    ),
                    "AUC_Gap": train_auc - val_auc,
                })

                progress.set_postfix(
                    trial=trial,
                    date=rebalance_date.strftime("%Y-%m-%d")
                )
                progress.update(1)

                del model

            del train_processed, val_processed, preprocessing

    # Summarize results by hyperparameter configuration
    search_by_rebalance = pd.DataFrame(search_daily_rows)

    parameter_table = pd.DataFrame([
        {
            "Trial": trial,
            **params,
            "Parameters_JSON": json.dumps(params, sort_keys=True)
        }
        for trial, params in enumerate(parameter_sets, start=1)
    ])

    hyperparameter_search_results = (
        search_by_rebalance
        .groupby("Trial", as_index=False)
        .agg(
            Validation_Dates=("Rebalance_Date", "nunique"),
            Val_ROC_AUC_Mean=("Val_ROC_AUC", "mean"),
            Val_ROC_AUC_SD=("Val_ROC_AUC", "std"),
            Valid_AUC_Dates=("Val_ROC_AUC", "count"),
            Val_AP_Mean=("Val_Average_Precision", "mean"),
            Val_Log_Loss_Mean=("Val_Log_Loss", "mean"),
            Train_ROC_AUC_Mean=("Train_ROC_AUC", "mean"),
            AUC_Gap_Mean=("AUC_Gap", "mean"),
        )
        .merge(parameter_table, on="Trial", validate="one_to_one")
        .sort_values(
            ["Val_ROC_AUC_Mean", "Val_AP_Mean", "AUC_Gap_Mean"],
            ascending=[False, False, True]
        )
        .reset_index(drop=True))

    assert (hyperparameter_search_results["Validation_Dates"] == len(walk_forward_splits)).all()
    assert hyperparameter_search_results["Valid_AUC_Dates"].nunique() == 1
    assert hyperparameter_search_results["Val_ROC_AUC_Mean"].notna().all()

    # Select best hyperparameters based on validation ROC-AUC
    best_trial = int(hyperparameter_search_results.loc[0, "Trial"])
    best_lr_params = dict(parameter_sets[best_trial - 1])

    hyperparameter_search_results["Selected"] = (
        hyperparameter_search_results["Trial"] == best_trial
    )

    best_hyperparameters = pd.DataFrame([{
        "Algorithm": "Logistic Regression",
        "Tuning_Specification": TUNING_SPECIFICATION,
        "Trial": best_trial,
        "Validation_Dates": len(walk_forward_splits),
        "Selection_Metric": "Mean within-date validation ROC AUC",
        "Val_ROC_AUC_Mean": hyperparameter_search_results.loc[
            0, "Val_ROC_AUC_Mean"
        ],
        "Val_AP_Mean": hyperparameter_search_results.loc[
            0, "Val_AP_Mean"
        ],
        **best_lr_params,
    }])

    print("\nSELECTED LOGISTIC REGRESSION CONFIGURATION")
    print(f"Best trial: {best_trial}")
    print(json.dumps(best_lr_params, indent=2, sort_keys=True))

    display(
        hyperparameter_search_results[[
            "Trial",
            "Selected",
            "Val_ROC_AUC_Mean",
            "Val_ROC_AUC_SD",
            "Val_AP_Mean",
            "Val_Log_Loss_Mean",
            "Train_ROC_AUC_Mean",
            "AUC_Gap_Mean",
            *LR_PARAM_SPACE.keys(),
        ]].round(4))

## Retrain tuned Logistic Regression and generate validation probabilities

In [7]:
prediction_parts = []
fit_rows = []

for rebalance_date, train_idx, val_idx in tqdm(
    walk_forward_splits,
    desc="Tuned Logistic Regression walk-forward",
    unit="rebalance",
):
    y_train = df.loc[train_idx, TARGET].to_numpy(dtype=int)
    y_val = df.loc[val_idx, TARGET].to_numpy(dtype=int)

    for algorithm in ALGORITHMS:
        for specification, features in SPECIFICATIONS.items():
            X_train = X(df.loc[train_idx, features], features)
            X_val = X(df.loc[val_idx, features], features)

            model = create_model()
            model.fit(X_train, y_train)
            train_p = predict_probabilities(model, X_train)
            val_p = predict_probabilities(model, X_val)

            for stage, y, p in [
                ("Train", y_train, train_p),
                ("Validation", y_val, val_p),
            ]:
                fit_rows.append({
                    "Rebalance_Date": rebalance_date,
                    "Algorithm": algorithm,
                    "Specification": specification,
                    "Stage": stage,
                    "Observations": len(y),
                    "Baseline_Precision": float(y.mean()),
                    "ROC_AUC": safe_auc(y, p),
                    "Average_Precision": safe_ap(y, p),
                    "Log_Loss": float(log_loss(y, p, labels=[0, 1])),
                    **{
                        f"Precision_{cutoff:.2f}": (
                            float(y[p >= cutoff].mean())
                            if np.any(p >= cutoff) else np.nan
                        )
                        for cutoff in config.CLASSIFICATION_THRESHOLDS
                    },
                    **{
                        f"Positive_Rate_{cutoff:.2f}": float(np.mean(p >= cutoff))
                        for cutoff in config.CLASSIFICATION_THRESHOLDS
                    },
                })

            part = df.loc[val_idx, ["Date", "Ticker", TARGET]].copy()
            part["Algorithm"] = algorithm
            part["Specification"] = specification
            part["Probability"] = val_p
            prediction_parts.append(part)
            del X_train, X_val, model

oof_predictions = pd.concat(prediction_parts, ignore_index=True)
train_validation_stage = pd.DataFrame(fit_rows)

assert oof_predictions["Probability"].between(0, 1).all()
assert not oof_predictions.duplicated(
    ["Date", "Ticker", "Algorithm", "Specification"]
).any()

print("Completed validation rebalances:", len(walk_forward_splits))
print("Final evaluation model fits:", len(train_validation_stage) // 2)
print("Validation predictions:", len(oof_predictions))

Tuned Logistic Regression walk-forward:   0%|          | 0/24 [00:00<?, ?rebalance/s]

Completed validation rebalances: 24
Final evaluation model fits: 48
Validation predictions: 2304


## Training vs validation

In [8]:
keys = ["Rebalance_Date", "Algorithm", "Specification"]

train_vals = train_validation_stage.pivot(index=keys, columns="Stage")
train_vals.columns = [f"{metric}_{stage}" for metric, stage in train_vals.columns]
train_vs_validation = train_vals.reset_index()

train_vs_validation["AUC_Gap"] = (
    train_vs_validation["ROC_AUC_Train"]
    - train_vs_validation["ROC_AUC_Validation"]
)
train_vs_validation["AP_Gap"] = (
    train_vs_validation["Average_Precision_Train"]
    - train_vs_validation["Average_Precision_Validation"]
)

train_validation_summary = (
    train_vs_validation
    .groupby(["Algorithm", "Specification"], as_index=False)[[
        "ROC_AUC_Train",
        "ROC_AUC_Validation",
        "AUC_Gap",
        "Average_Precision_Train",
        "Average_Precision_Validation",
        "AP_Gap",
        "Log_Loss_Train",
        "Log_Loss_Validation",
        "Precision_0.50_Train",
        "Precision_0.50_Validation",
        "Positive_Rate_0.50_Validation",
    ]]
    .mean()
)

display(train_validation_summary.sort_values(
    "ROC_AUC_Validation", ascending=False
).round(4))

,Algorithm,Specification,ROC_AUC_Train,ROC_AUC_Validation,AUC_Gap,Average_Precision_Train,Average_Precision_Validation,AP_Gap,Log_Loss_Train,Log_Loss_Validation,Precision_0.50_Train,Precision_0.50_Validation,Positive_Rate_0.50_Validation
1,Logistic Regression,Without Regimes,0.7933,0.6777,0.1156,0.8463,0.7458,0.1005,0.5404,0.7624,0.7536,0.6787,0.7196
0,Logistic Regression,With Regimes,0.7934,0.6776,0.1158,0.8464,0.7449,0.1015,0.5402,0.7640,0.7537,0.6807,0.7170


## Classification metrics and thresholds

In [9]:
direction_rows = []

# One pooled six-year result per model/specification/threshold
for (algorithm, specification), part in oof_predictions.groupby(
    ["Algorithm", "Specification"], sort=True
):
    for threshold in config.CLASSIFICATION_THRESHOLDS:
        direction_rows.append({
            "Algorithm": algorithm,
            "Specification": specification,
            "Threshold": threshold,
            **direction_metrics(part, threshold),
        })

# Always-positive benchmark, using each asset-date once
baseline_source = oof_predictions.loc[
    (oof_predictions["Algorithm"] == ALGORITHMS[0])
    & (oof_predictions["Specification"] == "Without Regimes")
]

for threshold in config.CLASSIFICATION_THRESHOLDS:
    direction_rows.append({
        "Algorithm": "AlwaysPositive",
        "Specification": "Baseline",
        "Threshold": threshold,
        **direction_metrics(
            baseline_source.assign(Probability=1.0),
            threshold
        ),
    })

direction_results = pd.DataFrame(direction_rows)

display(
    direction_results[[
        "Algorithm", "Specification", "Threshold",
        "Precision", "Always_Positive_Precision",
        "Precision_Lift_pp", "Recall", "Specificity",
        "Balanced_Accuracy", "Positive_Prediction_Rate",
        "ROC_AUC", "Average_Precision"
    ]]
    .sort_values(["Threshold", "Precision"], ascending=[True, False]).round(4))

,Algorithm,Specification,Threshold,Precision,Always_Positive_Precision,Precision_Lift_pp,Recall,Specificity,Balanced_Accuracy,Positive_Prediction_Rate,ROC_AUC,Average_Precision
6,AlwaysPositive,Baseline,0.50,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528
0,Logistic Regression,With Regimes,0.50,0.6356,0.6528,-1.7185,0.6981,0.2475,0.4728,0.7170,0.4853,0.6348
3,Logistic Regression,Without Regimes,0.50,0.6345,0.6528,-1.8278,0.6995,0.2425,0.4710,0.7196,0.4839,0.6346
7,AlwaysPositive,Baseline,0.55,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528
1,Logistic Regression,With Regimes,0.55,0.6317,0.6528,-2.1127,0.6157,0.3250,0.4703,0.6363,0.4853,0.6348
4,Logistic Regression,Without Regimes,0.55,0.6274,0.6528,-2.5381,0.6090,0.3200,0.4645,0.6337,0.4839,0.6346
8,AlwaysPositive,Baseline,0.60,0.6528,0.6528,0.0000,1.0000,0.0000,0.5000,1.0000,0.5000,0.6528
2,Logistic Regression,With Regimes,0.60,0.6315,0.6528,-2.1282,0.5332,0.4150,0.4741,0.5512,0.4853,0.6348
5,Logistic Regression,Without Regimes,0.60,0.6222,0.6528,-3.0556,0.5213,0.4050,0.4631,0.5469,0.4839,0.6346


## Probability-threshold and Hybrid Top-K screening

In [10]:
screening_daily_rows = []

for (algorithm, specification, date), daily in oof_predictions.groupby(
    ["Algorithm", "Specification", "Date"], sort=True
):
    ranked = daily.sort_values(["Probability", "Ticker"], ascending=[False, True], kind="stable")
    baseline = float(daily[TARGET].mean())

    choices = []
    for threshold in config.CLASSIFICATION_THRESHOLDS:
        eligible = ranked.loc[ranked["Probability"] >= threshold]

        # Threshold only: select all eligible assets
        choices.append(("Probability Threshold", threshold, eligible))

        # Hybrid: select at most K eligible assets
        for k in config.TOP_K_VALUES:
            choices.append((f"Hybrid Top {k}", threshold, eligible.head(k)))

    for rule, threshold, selected in choices:
        screening_daily_rows.append({
            "Date": date,
            "Algorithm": algorithm,
            "Specification": specification,
            "Rule": rule,
            "Threshold": threshold,
            "Available_Assets": len(daily),
            "Selected_Assets": len(selected),
            "Correct_Positives": int(selected[TARGET].sum()),
            "Same_Date_Baseline": baseline,
        })

screening_daily = pd.DataFrame(screening_daily_rows)


screening_rows = []

for (algorithm, specification, rule, threshold), part in screening_daily.groupby(
    ["Algorithm", "Specification", "Rule", "Threshold"],
    dropna=False,
    sort=True
):
    screening_rows.append({
        "Algorithm": algorithm,
        "Specification": specification,
        "Rule": rule,
        "Threshold": threshold,
        **screening_summary(part),
    })

screening_results = pd.DataFrame(screening_rows)

display(
    screening_results[[
        "Algorithm",
        "Specification",
        "Rule",
        "Threshold",
        "Dates",
        "Selected_Asset_Dates",
        "Precision",
        "Matched_Date_Baseline",
        "Precision_Lift_pp",
        "Mean_Assets",
        "Min_Assets",
        "Max_Assets",
        "Cash_Only_Dates"
    ]]
    .sort_values(["Rule", "Threshold", "Precision_Lift_pp"], ascending=[True, True, False]).round(4))

# Select the best validation strategy when tuning, otherwise use the saved model
if config.HYPERPARAMETER_TUNING:
    selection_candidates = screening_results.loc[screening_results["Mean_Assets"] >= 10].copy()
    assert not selection_candidates.empty, "No strategy meets the minimum coverage."
    selected_strategy = selection_candidates.sort_values(
        ["Precision_Lift_pp", "Mean_Assets"], ascending=[False, False]
    ).iloc[0]
else:
    saved_screening = loaded_config["screening"]
    matching = screening_results.loc[
        (screening_results["Algorithm"] == loaded_config["algorithm"])
        & (screening_results["Specification"] == loaded_config["specification"])
        & (screening_results["Rule"] == saved_screening["rule"])
        & np.isclose(
            screening_results["Threshold"].astype(float),
            float(saved_screening["threshold"]),
        )
    ]
    if len(matching) != 1:
        raise ValueError(
            "Saved screening rule/specification not found in current validation results. "
            "Check CLASSIFICATION_THRESHOLDS, TOP_K_VALUES, and selected_config.json."
        )
    selected_strategy = matching.iloc[0]
    print("Reusing the saved screening rule (no re-selection on validation or test data).")

STRATEGY_RESULT_COLUMNS = [
    "Date", "Algorithm", "Specification", "Rule", "Threshold",
    "Train_Obs", "Available_Assets", "Selected_Assets", "Correct_Positives",
    "Precision", "Baseline", "Precision_Lift_pp",
    "Ticker", "Probability", "Predicted_Sign", "Actual_Sign",
    "Forward_Return_60d", "Outcomes_Available",
]

# Asset-level results for the selected screening strategy
selected_assets = oof_predictions.loc[
    (oof_predictions["Algorithm"] == selected_strategy["Algorithm"])
    & (oof_predictions["Specification"] == selected_strategy["Specification"])
    & (oof_predictions["Probability"] >= selected_strategy["Threshold"])
].copy()

selected_assets = selected_assets.sort_values(["Date", "Probability", "Ticker"], ascending=[True, False, True])

# Apply Top-K limit only if the selected rule is Hybrid
selection_rule = str(selected_strategy["Rule"])

if selection_rule.startswith("Hybrid Top"):
    k = int(selection_rule.split()[-1])
    selected_assets = selected_assets.groupby("Date", group_keys=False).head(k)

selected_assets["Predicted_Sign"] = "Positive"
selected_assets["Actual_Sign"] = np.where(
    selected_assets[TARGET] == 1, "Positive", "Negative"
)

# Use the same asset-level output schema for validation and test results
# Pull the daily totals directly from the selected screening rule, retaining cash-only dates
selected_daily = screening_daily.loc[
    (screening_daily["Algorithm"] == selected_strategy["Algorithm"])
    & (screening_daily["Specification"] == selected_strategy["Specification"])
    & (screening_daily["Rule"] == selected_strategy["Rule"])
    & np.isclose(
        screening_daily["Threshold"].astype(float),
        float(selected_strategy["Threshold"]),
    )
].copy()
assert len(selected_daily) == len(walk_forward_splits)

# Training observations per rebalance are already recorded in the validation schedule
train_count_by_date = pd.DataFrame(validation_summary).set_index("Rebalance_Date")["Train_Obs"]
selected_daily["Train_Obs"] = selected_daily["Date"].map(train_count_by_date)
assert selected_daily["Train_Obs"].notna().all()
selected_daily["Precision"] = (
    selected_daily["Correct_Positives"]
    / selected_daily["Selected_Assets"].replace(0, np.nan)
)
selected_daily["Precision_Lift_pp"] = (
    100.0 * (selected_daily["Precision"] - selected_daily["Same_Date_Baseline"])
)
selected_daily["Outcomes_Available"] = True

# Add the observed 60-session return for selected assets only, validation labels are mature
validation_selected_assets = selected_assets[[
    "Date", "Ticker", "Probability", "Predicted_Sign", "Actual_Sign"
]].merge(
    df[["Date", "Ticker", "Forward_Return_60d"]],
    on=["Date", "Ticker"],
    how="left",
    validate="one_to_one",
)

selected_strategy_details = selected_daily.merge(
    validation_selected_assets,
    on="Date", how="left", validate="one_to_many",
).rename(columns={"Same_Date_Baseline": "Baseline"})
selected_strategy_details = selected_strategy_details[STRATEGY_RESULT_COLUMNS]
assert selected_strategy_details["Outcomes_Available"].all()

print("\nSELECTED SCREENING STRATEGY")
print(f"Algorithm: {selected_strategy['Algorithm']}")
print(f"Specification: {selected_strategy['Specification']}")
print(f"Rule: {selected_strategy['Rule']}")
print(f"Threshold: {selected_strategy['Threshold']:.2f}")
print(f"Precision: {selected_strategy['Precision']:.2%}")
print(f"Precision lift: {selected_strategy['Precision_Lift_pp']:.2f} pp")
print(f"Mean selected assets: {selected_strategy['Mean_Assets']:.2f}")
print(f"Min selected assets: {int(selected_strategy['Min_Assets'])}")
print(f"Max selected assets: {int(selected_strategy['Max_Assets'])}")
print(f"Cash-only dates: {int(selected_strategy['Cash_Only_Dates'])}")

,Algorithm,Specification,Rule,Threshold,Dates,Selected_Asset_Dates,Precision,Matched_Date_Baseline,Precision_Lift_pp,Mean_Assets,Min_Assets,Max_Assets,Cash_Only_Dates
0,Logistic Regression,With Regimes,Hybrid Top 10,0.50,24,216,0.6991,0.6249,7.4171,9.0000,0,10,1
9,Logistic Regression,Without Regimes,Hybrid Top 10,0.50,24,213,0.6948,0.6214,7.3455,8.8750,0,10,1
10,Logistic Regression,Without Regimes,Hybrid Top 10,0.55,24,192,0.6875,0.6128,7.4653,8.0000,0,10,3
1,Logistic Regression,With Regimes,Hybrid Top 10,0.55,24,190,0.6842,0.6138,7.0395,7.9167,0,10,3
2,Logistic Regression,With Regimes,Hybrid Top 10,0.60,24,176,0.6648,0.6075,5.7292,7.3333,0,10,5
11,Logistic Regression,Without Regimes,Hybrid Top 10,0.60,24,172,0.6628,0.6083,5.4506,7.1667,0,10,5
3,Logistic Regression,With Regimes,Hybrid Top 15,0.50,24,321,0.7009,0.6227,7.8271,13.3750,0,15,1
12,Logistic Regression,Without Regimes,Hybrid Top 15,0.50,24,318,0.6981,0.6203,7.7830,13.2500,0,15,1
13,Logistic Regression,Without Regimes,Hybrid Top 15,0.55,24,282,0.6809,0.6116,6.9297,11.7500,0,15,3
4,Logistic Regression,With Regimes,Hybrid Top 15,0.55,24,280,0.6786,0.6122,6.6369,11.6667,0,15,3


Reusing the saved screening rule (no re-selection on validation or test data).

SELECTED SCREENING STRATEGY
Algorithm: Logistic Regression
Specification: With Regimes
Rule: Hybrid Top 15
Threshold: 0.50
Precision: 70.09%
Precision lift: 7.83 pp
Mean selected assets: 13.38
Min selected assets: 0
Max selected assets: 15
Cash-only dates: 1


## Test the selected strategy

In [11]:
# Freeze the Logistic Regression parameters and screening strategy selected on validation
# At each test rebalance, refit on labels known strictly before that date
# No test-period labels are used for hyperparameter or screening-rule selection

TEST_END = pd.Timestamp(getattr(config, "TEST_END", "2026-12-31"))
if TEST_END < TEST_START:
    raise ValueError("TEST_END must be on or after TEST_START.")

test_positions = np.arange(anchor_pos, len(trading_dates), REBALANCE_STEP)
test_dates = trading_dates[test_positions]
test_dates = test_dates[(test_dates >= TEST_START) & (test_dates <= TEST_END)]

selected_features = SPECIFICATIONS[str(selected_strategy["Specification"])]
test_rule = str(selected_strategy["Rule"])
test_threshold = float(selected_strategy["Threshold"])
test_top_k = int(test_rule.split()[-1]) if test_rule.startswith("Hybrid Top") else None

last_observed_date = trading_dates[-1]
test_selection_parts = []
test_rebalance_rows = []
test_prediction_parts = []

for rebalance_date in tqdm(test_dates, desc="Logistic Regression test walk-forward", unit="rebalance"):
    # Use indices to avoid retaining copies of every expanding training window
    train_idx = df.index[
        (df["Date"] < rebalance_date)
        & (df["_Target_End_Date"] < rebalance_date)
        & valid_target
    ]
    candidate_idx = df.index[df["Date"] == rebalance_date]
    if len(train_idx) == 0 or len(candidate_idx) == 0:
        continue
    y_train = df.loc[train_idx, TARGET].to_numpy(dtype=int)
    if np.unique(y_train).size != 2:
        raise ValueError(f"Training targets have only one class at {rebalance_date}.")
    assert df.loc[train_idx, "_Target_End_Date"].max() < rebalance_date

    X_train = X(df.loc[train_idx, selected_features], selected_features)
    X_candidates = X(df.loc[candidate_idx, selected_features], selected_features)

    model = create_model()  # Selected C and training-only preprocessing
    model.fit(X_train, y_train)
    candidate_p = predict_probabilities(model, X_candidates)

    candidates = df.loc[candidate_idx, [
        "Date", "Ticker", TARGET, "Forward_Return_60d", "_Target_End_Date"
    ]].copy()
    candidates["Probability"] = candidate_p
    ranked = candidates.sort_values(["Probability", "Ticker"], ascending=[False, True], kind="stable")

    # Require the entire same-date universe to have complete 60-session outcomes
    complete_outcomes = bool((
        candidates[TARGET].isin([0, 1])
        & candidates["Forward_Return_60d"].notna()
        & candidates["_Target_End_Date"].notna()
        & (candidates["_Target_End_Date"] <= last_observed_date)
    ).all())

    # Save full-universe probabilities, not just selected Top-K assets
    all_candidates = ranked[["Date", "Ticker", TARGET, "Probability", "Forward_Return_60d"]].copy()
    all_candidates["Outcomes_Available"] = complete_outcomes
    if not complete_outcomes:
        all_candidates[TARGET] = np.nan
        all_candidates["Forward_Return_60d"] = np.nan
    test_prediction_parts.append(all_candidates)

    eligible = ranked.loc[ranked["Probability"] >= test_threshold]
    selected = eligible.head(test_top_k).copy() if test_top_k is not None else eligible.copy()

    n_selected = len(selected)
    n_correct = int(selected[TARGET].sum()) if complete_outcomes else np.nan
    daily_precision = n_correct / n_selected if complete_outcomes and n_selected else np.nan
    baseline = float(candidates[TARGET].mean()) if complete_outcomes else np.nan
    daily_lift = 100.0 * (daily_precision - baseline) if complete_outcomes and n_selected else np.nan

    test_rebalance_rows.append({
        "Date": rebalance_date,
        "Algorithm": "Logistic Regression",
        "Specification": str(selected_strategy["Specification"]),
        "Rule": test_rule,
        "Threshold": test_threshold,
        "Train_Obs": len(train_idx),
        "Available_Assets": len(candidates),
        "Selected_Assets": n_selected,
        "Correct_Positives": n_correct,
        "Precision": daily_precision,
        "Same_Date_Baseline": baseline,
        "Precision_Lift_pp": daily_lift,
        "Outcomes_Available": complete_outcomes,
    })

    if n_selected:
        selected["Predicted_Sign"] = "Positive"
        selected["Actual_Sign"] = (
            np.where(selected[TARGET] == 1, "Positive", "Negative")
            if complete_outcomes else pd.NA
        )
        if not complete_outcomes:
            selected["Forward_Return_60d"] = np.nan
        selected["Outcomes_Available"] = complete_outcomes
        test_selection_parts.append(selected[[
            "Date", "Ticker", "Probability", "Predicted_Sign",
            "Actual_Sign", "Forward_Return_60d", "Outcomes_Available"
        ]])
    del X_train, X_candidates, model

if not test_rebalance_rows:
    raise ValueError("No test rebalances evaluated. Check test dates and feature data.")

test_rebalance_summary = pd.DataFrame(test_rebalance_rows)
test_asset_columns = [
    "Date", "Ticker", "Probability", "Predicted_Sign", "Actual_Sign",
    "Forward_Return_60d", "Outcomes_Available",
]
test_selected_assets = (
    pd.concat(test_selection_parts, ignore_index=True)
    if test_selection_parts else pd.DataFrame(columns=test_asset_columns)
)

selected_strategy_test_results = test_rebalance_summary.drop(
    columns="Outcomes_Available"
).merge(test_selected_assets, on="Date", how="left", validate="one_to_many")
selected_strategy_test_results["Outcomes_Available"] = (
    selected_strategy_test_results["Outcomes_Available"].fillna(
        selected_strategy_test_results["Date"].map(
            test_rebalance_summary.set_index("Date")["Outcomes_Available"]
        )
    ).astype(bool)
)


# Align the test strategy export with the validation asset-level schema
selected_strategy_test_results = selected_strategy_test_results.rename(
    columns={"Same_Date_Baseline": "Baseline"}
)[STRATEGY_RESULT_COLUMNS]

scored = test_rebalance_summary.loc[test_rebalance_summary["Outcomes_Available"]].copy()
scored_selected = int(scored["Selected_Assets"].sum())
scored_correct = int(scored["Correct_Positives"].sum()) if not scored.empty else 0
scored_precision = scored_correct / scored_selected if scored_selected else np.nan
matched_baseline = (
    float((scored["Selected_Assets"] * scored["Same_Date_Baseline"]).sum() / scored_selected)
    if scored_selected else np.nan
)
scored_lift = 100.0 * (scored_precision - matched_baseline) if scored_selected else np.nan

print("\nUNSEEN TEST RESULTS:")
print(f"Scheduled test rebalances: {len(test_dates)}")
print(f"Completed test rebalances: {len(test_rebalance_summary)}")
print(f"Fully matured test rebalances: {len(scored)}")
print(f"Pending test rebalances: {len(test_rebalance_summary) - len(scored)}")
print(f"Selected asset-dates (matured): {scored_selected}")
print(f"Test precision (matured): {scored_precision:.2%}")
print(f"Matched baseline (matured): {matched_baseline:.2%}")
print(f"Test precision lift (matured): {scored_lift:.2f} pp")
print(f"Mean selected assets per evaluated date: {test_rebalance_summary['Selected_Assets'].mean():.2f}")
print(f"Cash-only dates: {int((test_rebalance_summary['Selected_Assets'] == 0).sum())}")
if len(scored) == 0:
    print("No mature test outcomes available; selections exported without outcome metrics.")

# Full-universe test ROC-AUC and average precision (not screening-selected assets only)
test_all_predictions = pd.concat(test_prediction_parts, ignore_index=True)
test_scored_predictions = test_all_predictions.loc[
    test_all_predictions["Outcomes_Available"]
].copy()

classification_by_date = []
for date, part in test_scored_predictions.groupby("Date"):
    y = part[TARGET].to_numpy(dtype=int)
    p = part["Probability"].to_numpy(dtype=float)
    classification_by_date.append({
        "Date": date,
        "Observations": len(part),
        "Baseline_Precision": float(y.mean()),
        "ROC_AUC": safe_auc(y, p),
        "Average_Precision": safe_ap(y, p),
    })

test_classification_by_rebalance = pd.DataFrame(classification_by_date, columns=[
    "Date", "Observations", "Baseline_Precision", "ROC_AUC", "Average_Precision"
])

if not test_scored_predictions.empty:
    test_y = test_scored_predictions[TARGET].to_numpy(dtype=int)
    test_p = test_scored_predictions["Probability"].to_numpy(dtype=float)
    pooled_auc = safe_auc(test_y, test_p)
    pooled_ap = safe_ap(test_y, test_p)
else:
    pooled_auc = np.nan
    pooled_ap = np.nan

test_classification_metrics = pd.DataFrame([{
    "Algorithm": "Logistic Regression",
    "Specification": str(selected_strategy["Specification"]),
    "Observations": len(test_scored_predictions),
    "Scored_Dates": test_scored_predictions["Date"].nunique(),
    "Pooled_ROC_AUC": pooled_auc,
    "Pooled_Average_Precision": pooled_ap,
    "Mean_ROC_AUC": test_classification_by_rebalance["ROC_AUC"].mean(),
    "Mean_Average_Precision": test_classification_by_rebalance["Average_Precision"].mean(),
}])

print("\nUNSEEN TEST CLASSIFICATION METRICS:")
display(test_classification_metrics.round(4))
print(f"Test mean within-date ROC-AUC: {test_classification_metrics.loc[0, 'Mean_ROC_AUC']:.4f}")
print(f"Test pooled ROC-AUC: {test_classification_metrics.loc[0, 'Pooled_ROC_AUC']:.4f}")

Frozen Logistic Regression test walk-forward:   0%|          | 0/19 [00:00<?, ?rebalance/s]


UNSEEN TEST RESULTS
Scheduled test rebalances: 19
Completed test rebalances: 19
Fully matured test rebalances: 18
Pending test rebalances: 1
Selected asset-dates (matured): 213
Test precision (matured): 75.59%
Matched baseline (matured): 67.61%
Test precision lift (matured): 7.98 pp
Mean selected assets per evaluated date: 12.00
Cash-only dates: 3

UNSEEN TEST CLASSIFICATION METRICS


,Algorithm,Specification,Observations,Scored_Dates,Pooled_ROC_AUC,Pooled_Average_Precision,Mean_ROC_AUC,Mean_Average_Precision
0,Logistic Regression,With Regimes,864,18,0.6272,0.6831,0.6081,0.7115


Test mean within-date ROC-AUC: 0.6081
Test pooled ROC-AUC: 0.6272


## Checks

In [12]:
assert oof_predictions["Date"].nunique() == len(walk_forward_splits)
if config.HYPERPARAMETER_TUNING:
    assert len(hyperparameter_search_results) == N_ITER
else:
    assert loaded_config is not None
    assert best_lr_params == loaded_config["model_params"]
assert oof_predictions[["Algorithm", "Specification"]].drop_duplicates().shape[0] == 2
assert oof_predictions["Probability"].between(0, 1).all()
assert len(direction_results) == 3 * len(config.CLASSIFICATION_THRESHOLDS)
assert len(screening_results) == 2 * len(config.CLASSIFICATION_THRESHOLDS) * (1 + len(config.TOP_K_VALUES))

assert not test_rebalance_summary["Date"].duplicated().any()
assert test_rebalance_summary["Date"].ge(TEST_START).all()
assert test_rebalance_summary["Date"].le(TEST_END).all()
assert selected_strategy_test_results["Probability"].dropna().between(0, 1).all()
assert selected_strategy_test_results["Rule"].eq(test_rule).all()
assert selected_strategy_test_results["Specification"].eq(str(selected_strategy["Specification"])).all()
assert test_all_predictions["Probability"].between(0, 1).all()
assert not test_all_predictions.duplicated(["Date", "Ticker"]).any()
print("All validation and test checks passed.")

assert selected_strategy_details.columns.tolist() == STRATEGY_RESULT_COLUMNS
assert selected_strategy_test_results.columns.tolist() == STRATEGY_RESULT_COLUMNS
assert not selected_strategy_details.dropna(subset=["Ticker"]).duplicated(["Date", "Ticker"]).any()
assert not selected_strategy_test_results.dropna(subset=["Ticker"]).duplicated(["Date", "Ticker"]).any()
assert selected_strategy_details["Probability"].dropna().between(0, 1).all()
assert selected_strategy_test_results["Probability"].dropna().between(0, 1).all()
assert test_classification_by_rebalance["Date"].nunique() == test_rebalance_summary["Outcomes_Available"].sum()


All validation and test checks passed.


## Export results

In [13]:
# Export validation and test results to the Logistic Regression output folder
csv_files = {
    "validation_classification_metrics.csv": direction_results,
    "train_validation_metrics.csv": train_validation_summary,
    "train_validation_by_rebalance.csv": train_vs_validation,
    "screening_metrics.csv": screening_results,
    "selected_strategy_validation_results.csv": selected_strategy_details,
    "selected_strategy_test_results.csv": selected_strategy_test_results,
    "test_classification_metrics.csv": test_classification_metrics,
    "test_classification_by_rebalance.csv": test_classification_by_rebalance,
}

if config.HYPERPARAMETER_TUNING:
    csv_files.update({
        "hyperparameter_search.csv": hyperparameter_search_results,
        "best_hyperparameters.csv": best_hyperparameters,
    })

for filename, frame in csv_files.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

if config.HYPERPARAMETER_TUNING:
    # Export selected model and screening settings
    rule = str(selected_strategy["Rule"])
    model_params = create_classifier(params=best_lr_params).get_params()

    selected_config = {
        "algorithm": str(selected_strategy["Algorithm"]),
        "specification": str(selected_strategy["Specification"]),
        "model_params": {
            "C": model_params["C"],
            "solver": model_params["solver"],
            "l1_ratio": model_params["l1_ratio"],
            "max_iter": model_params["max_iter"],
            "random_state": model_params["random_state"],
        },
        "preprocessing": {
            "numerical": "median imputation + StandardScaler",
            "indicators": "passthrough",
            "fitted_on": "training window only",
        },
        "screening": {
            "rule": rule,
            "threshold": float(selected_strategy["Threshold"]),
            "top_k": int(rule.split()[-1]) if rule.startswith("Hybrid Top") else None,
        },
    }
    with selected_config_path.open("w", encoding="utf-8") as f:
        json.dump(
            selected_config, f, indent=4,
            default=lambda x: x.item() if isinstance(x, np.generic) else str(x),
        )
    print(f"Validation results and selected configuration saved to {OUTPUT_DIR}/")
else:
    print(f"Validation and test results saved to {OUTPUT_DIR}/")

Validation and test results saved to classification_results\logistic_regression/
